In [16]:
import os
import random
import shutil
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter, ImageOps

Modyfikacja paneli, samych paneli

In [17]:
IMAGE_DIR = "obrazy"
OUTPUT_IMG_DIR = "aug/obrazy"
OUTPUT_LABEL_DIR = "aug/etykiety"
AUG_PER_IMAGE = 100
CLASS_ID = 0  # zakładamy że to jest zawsze ten sam obiekt
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)

In [18]:
def apply_scaling(img, abs_boxes):
    width, height = img.size
    scale = random.uniform(0.5, 1.5)
    img = img.resize((int(width * scale), int(height * scale)), Image.ANTIALIAS)
    scale_x = img.size[0] / width
    scale_y = img.size[1] / height
    abs_boxes = [
        [cls, int(x1 * scale_x), int(y1 * scale_y), int(x2 * scale_x), int(y2 * scale_y)]
        for cls, x1, y1, x2, y2 in abs_boxes
    ]
    return img, abs_boxes

In [19]:
def apply_rotation(img):
    angle = random.uniform(-30, 30)
    img = img.convert("RGBA")
    img = img.rotate(angle, expand=True)
    img = img.convert("RGB")
    return img

In [20]:

def apply_blur(img):
    if random.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0, 2)))
    return img


In [21]:
def apply_brightness(img):
    if random.random() < 0.7:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.6, 1.4))
    return img

In [22]:
def apply_contrast(img):
    if random.random() < 0.7:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.6, 1.4))
    return img

In [23]:
def apply_color(img):
    if random.random() < 0.5:
        img = ImageEnhance.Color(img).enhance(random.uniform(0.6, 1.4))
    return img

In [24]:
def apply_crop(img, abs_boxes):
    if random.random() < 0.5:
        crop_x = random.randint(0, int(img.size[0] * 0.1))
        crop_y = random.randint(0, int(img.size[1] * 0.1))
        crop_w = img.size[0] - crop_x
        crop_h = img.size[1] - crop_y
        img = img.crop((crop_x, crop_y, crop_w, crop_h))
        abs_boxes = [
            [cls, x1 - crop_x, y1 - crop_y, x2 - crop_x, y2 - crop_y]
            for cls, x1, y1, x2, y2 in abs_boxes
        ]
    return img, abs_boxes

In [25]:
def apply_flip(img, abs_boxes):
    if random.random() < 0.5:
        img = ImageOps.mirror(img)
        w = img.size[0]
        abs_boxes = [
            [cls, w - x2, y1, w - x1, y2] for cls, x1, y1, x2, y2 in abs_boxes
        ]
    return img, abs_boxes

In [26]:
def apply_padding(img, abs_boxes):
    if random.random() < 0.5:
        pad = random.randint(10, 50)
        img = ImageOps.expand(img, border=pad, fill=(0, 0, 0))
        abs_boxes = [
            [cls, x1 + pad, y1 + pad, x2 + pad, y2 + pad] for cls, x1, y1, x2, y2 in abs_boxes
        ]
    return img, abs_boxes

In [27]:
def apply_noise(img):
    if random.random() < 0.5:
        np_img = np.array(img).astype(np.int16)
        noise = np.random.normal(0, 25, np_img.shape)
        np_img = np.clip(np_img + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(np_img)
    return img

In [28]:
def transform_image_and_boxes(img, abs_boxes):
    img, abs_boxes = apply_scaling(img, abs_boxes)
    img = apply_rotation(img)
    img = apply_blur(img)
    img = apply_brightness(img)
    img = apply_contrast(img)
    img = apply_color(img)
    img, abs_boxes = apply_crop(img, abs_boxes)
    img, abs_boxes = apply_flip(img, abs_boxes)
    img, abs_boxes = apply_padding(img, abs_boxes)
    img = apply_noise(img)
    return img, abs_boxes

In [29]:
def get_bbox_for_image(img):
    w, h = img.size
    xc = 0.5
    yc = 0.5
    bw = 1.0
    bh = 1.0
    return [[CLASS_ID, xc, yc, bw, bh]]

def save_yolo_labels(label_path, boxes):
    with open(label_path, 'w') as f:
        for box in boxes:
            cls, xc, yc, w, h = box
            f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

def denormalize_boxes(boxes, width, height):
    abs_boxes = []
    for cls, xc, yc, w, h in boxes:
        abs_boxes.append([
            cls,
            int((xc - w / 2) * width),
            int((yc - h / 2) * height),
            int((xc + w / 2) * width),
            int((yc + h / 2) * height)
        ])
    return abs_boxes

def normalize_boxes(boxes, width, height):
    norm_boxes = []
    for cls, x1, y1, x2, y2 in boxes:
        x1 = max(0, min(x1, width))
        y1 = max(0, min(y1, height))
        x2 = max(0, min(x2, width))
        y2 = max(0, min(y2, height))
        xc = (x1 + x2) / 2 / width
        yc = (y1 + y2) / 2 / height
        w = (x2 - x1) / width
        h = (y2 - y1) / height
        if w > 0 and h > 0:
            norm_boxes.append([cls, xc, yc, w, h])
    return norm_boxes

In [31]:
for file in os.listdir(IMAGE_DIR):
    if not file.lower().endswith(('.jpg', '.png')):
        continue

    img_path = os.path.join(IMAGE_DIR, file)
    original_img = Image.open(img_path)
    boxes = get_bbox_for_image(original_img)

    for i in range(AUG_PER_IMAGE):
        img = original_img.copy()
        abs_boxes = denormalize_boxes(boxes, img.width, img.height)
        aug_img, aug_boxes = transform_image_and_boxes(img, abs_boxes)
        norm_boxes = normalize_boxes(aug_boxes, aug_img.width, aug_img.height)

        out_name = f"{os.path.splitext(file)[0]}_aug_{i:03d}"
        aug_img.save(os.path.join(OUTPUT_IMG_DIR, out_name + ".jpg"))
        save_yolo_labels(os.path.join(OUTPUT_LABEL_DIR, out_name + ".txt"), norm_boxes)

    print(f"✔ {file} - {AUG_PER_IMAGE} augmentacji")

AttributeError: module 'PIL.Image' has no attribute 'ANTIALIAS'